# Project 3: Computer Vision — HOG + SVM Digit Classifier

From-scratch: HOG feature extraction, SVM with RBF kernel, full evaluation. No deep learning frameworks.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
print("Dataset: sklearn digits")
print("Shape:", X.shape, "  Classes:", np.unique(y))
print("Pixels per image: 8x8 =", X.shape[1])
print("Samples per class:")
for c in np.unique(y):
    print("  digit {}: {}".format(c, (y==c).sum()))


Dataset: sklearn digits
Shape: (1797, 64)   Classes: [0 1 2 3 4 5 6 7 8 9]
Pixels per image: 8x8 = 64
Samples per class:
  digit 0: 178
  digit 1: 182
  digit 2: 177
  digit 3: 183
  digit 4: 181
  digit 5: 182
  digit 6: 181
  digit 7: 179
  digit 8: 174
  digit 9: 180


In [2]:
def hog_features(images_flat, img_size=8, cell_size=2, nbins=9):
    features = []
    for img_flat in images_flat:
        img = img_flat.reshape(img_size, img_size)
        gx = np.gradient(img, axis=1)
        gy = np.gradient(img, axis=0)
        mag = np.sqrt(gx**2 + gy**2)
        ang = np.arctan2(gy, gx) * (180 / np.pi) % 180
        cells_y = img_size // cell_size
        cells_x = img_size // cell_size
        hog = []
        for cy in range(cells_y):
            for cx in range(cells_x):
                m = mag[cy*cell_size:(cy+1)*cell_size, cx*cell_size:(cx+1)*cell_size]
                a = ang[cy*cell_size:(cy+1)*cell_size, cx*cell_size:(cx+1)*cell_size]
                hist, _ = np.histogram(a, bins=nbins, range=(0,180), weights=m)
                norm = hist / (np.linalg.norm(hist) + 1e-6)
                hog.extend(norm)
        features.append(hog)
    return np.array(features)

X_hog = hog_features(X)
print("HOG features shape:", X_hog.shape)
print("Feature stats — mean: {:.4f}, std: {:.4f}".format(X_hog.mean(), X_hog.std()))


HOG features shape: (1797, 144)
Feature stats — mean: 0.1486, std: 0.2895


In [3]:
X_combined = np.hstack([X / 16.0, X_hog])
print("Combined feature vector:", X_combined.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=0.99, svd_solver="full")),
    ("svm",    SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42))
])

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)
proba = pipeline.predict_proba(X_test)

pca_step = pipeline.named_steps["pca"]
print("PCA kept components:", pca_step.n_components_)
print("Explained variance: {:.4f}".format(pca_step.explained_variance_ratio_.sum()))
print("\n=== SVM + HOG Results ===")
print(classification_report(y_test, preds))
print("Overall Accuracy: {:.4f}".format(accuracy_score(y_test, preds)))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))


Combined feature vector: (1797, 208)


PCA kept components: 172
Explained variance: 0.9901

=== SVM + HOG Results ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.97      0.97      0.97        36
           2       0.97      0.97      0.97        35
           3       0.95      0.97      0.96        37
           4       0.97      0.97      0.97        36
           5       1.00      0.97      0.99        37
           6       1.00      1.00      1.00        36
           7       0.95      1.00      0.97        36
           8       0.97      0.94      0.96        35
           9       1.00      0.97      0.99        36

    accuracy                           0.98       360
   macro avg       0.98      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360

Overall Accuracy: 0.9778

Confusion Matrix:
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  0  0  1  0  0  0  0  0]
 [ 0  0 34  1  0  0  0  0  0  0]
 [ 0  0  1 36  0  0

In [4]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

baselines = {
    "Raw pixels + SVM"  : Pipeline([("sc", StandardScaler()),
                                     ("svm", SVC(kernel="rbf", C=10, gamma="scale"))]),
    "HOG + SVM"         : pipeline,
    "HOG + KNN(k=5)"    : Pipeline([("sc", StandardScaler()),
                                     ("knn", KNeighborsClassifier(n_neighbors=5))]),
    "HOG + RandomForest": Pipeline([("sc", StandardScaler()),
                                     ("rf", RandomForestClassifier(n_estimators=200, random_state=42))]),
}
X_raw_comb = np.hstack([X / 16.0, X_hog])
X_tr2, X_te2, yr2, ye2 = train_test_split(X_raw_comb, y, test_size=0.2, stratify=y, random_state=42)

print("Model Comparison:")
for name, model in baselines.items():
    model.fit(X_tr2, yr2)
    acc = accuracy_score(ye2, model.predict(X_te2))
    print("  {:25s}  Acc={:.4f}".format(name, acc))

# Per-class accuracy
print("\nPer-digit accuracy (best model):")
cm = confusion_matrix(y_test, preds)
for i in range(10):
    per_acc = cm[i,i] / cm[i].sum()
    bar = "#" * int(per_acc * 30)
    print("  digit {}:  {:.4f}  {}".format(i, per_acc, bar))


Model Comparison:


  Raw pixels + SVM           Acc=0.9778


  HOG + SVM                  Acc=0.9778
  HOG + KNN(k=5)             Acc=0.9556


  HOG + RandomForest         Acc=0.9750

Per-digit accuracy (best model):
  digit 0:  1.0000  ##############################
  digit 1:  0.9722  #############################
  digit 2:  0.9714  #############################
  digit 3:  0.9730  #############################
  digit 4:  0.9722  #############################
  digit 5:  0.9730  #############################
  digit 6:  1.0000  ##############################
  digit 7:  1.0000  ##############################
  digit 8:  0.9429  ############################
  digit 9:  0.9722  #############################
